<a href="https://colab.research.google.com/github/buabns/Aulas_PCD/blob/main/PROJETO_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Importação de Bibliotecas e Dados

In [ ]:
# Bibliotecas utilizadas no projeto
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.neighbors import NearestNeighbors

# Caminho do arquivo disponibilizado no Google Drive
caminho_arquivo = "/content/drive/MyDrive/steam-200k.csv.zip"

# Monta o Google Drive quando o notebook está sendo executado no Colab.
# Fora do Colab, o caminho do arquivo precisa estar disponível no ambiente.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Ambiente fora do Google Colab: verifique o caminho do arquivo.")

# O arquivo é lido sem cabeçalho, atribuindo nomes às cinco colunas esperadas.
# A coluna adicional 'other' não é utilizada nas análises e é descartada em seguida.
df = pd.read_csv(
    caminho_arquivo,
    header=None,
    names=['user_id', 'game_name', 'behavior', 'hours', 'other'],
    low_memory=False
)
df = df.drop(columns=['other'])

print("Colunas utilizadas:", df.columns.tolist())
print("Dimensão inicial do conjunto de dados:", df.shape)
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colunas utilizadas: ['user_id', 'game_name', 'behavior', 'hours']
Dimensão inicial do conjunto de dados: (200000, 4)


,user_id,game_name,behavior,hours
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0
1,151603712,The Elder Scrolls V Skyrim,play,273.0
2,151603712,Fallout 4,purchase,1.0
3,151603712,Fallout 4,play,87.0
4,151603712,Spore,purchase,1.0


# 2. Limpeza e Filtragem dos Dados

In [ ]:
# Mantém somente os registros em que o usuário efetivamente jogou o título.
# Registros de compra são descartados porque o projeto analisa horas jogadas.
df_play = df[df['behavior'] == 'play'].copy()

# Após o filtro, a coluna behavior contém apenas 'play' e deixa de ser necessária.
df_play = df_play.drop(columns=['behavior'])

print("Tamanho após filtrar os registros 'play':", df_play.shape)
display(df_play.head())

Tamanho após filtrar os registros 'play': (70489, 3)


,user_id,game_name,hours
1,151603712,The Elder Scrolls V Skyrim,273.0
3,151603712,Fallout 4,87.0
5,151603712,Spore,14.9
7,151603712,Fallout New Vegas,12.1
9,151603712,Left 4 Dead 2,8.9


In [ ]:
# Verifica a quantidade de valores ausentes em cada coluna.
# Esta etapa identifica os valores nulos; não realiza uma remoção geral de NaN.
print("\nValores nulos por coluna:")
print(df_play.isnull().sum())

In [ ]:
# Mantém somente as interações com pelo menos 0,5 hora (30 minutos) de jogo.
# O filtro remove registros abaixo do limite, não usuários inteiros.
limite_minimo_horas = 0.5
df_play = df_play[df_play['hours'] >= limite_minimo_horas].copy()

print("Tamanho após aplicar o limite mínimo de horas:", df_play.shape)

In [ ]:
# Remove linhas completamente duplicadas, evitando contar duas vezes
# registros idênticos nas etapas posteriores.
df_play = df_play.drop_duplicates()

print("Tamanho após remover registros duplicados:", df_play.shape)
display(df_play.head())

# 3. Transformação Logarítmica das Horas Jogadas

In [ ]:
# Aplica log(1 + horas) aos tempos de jogo.
# A transformação comprime valores muito altos e reduz a assimetria da escala,
# diminuindo a influência relativa de registros com milhares de horas.
# log1p também permite calcular a transformação quando o valor é zero.
df_play['hours_weight'] = np.log1p(df_play['hours'])

display(df_play[['user_id', 'game_name', 'hours', 'hours_weight']].head())

# 4. Análise Exploratória e Visualização de Dados

In [ ]:
# Define o estilo visual utilizado nos gráficos do projeto.
sns.set_theme(style="darkgrid", palette="viridis")

# Conta quantos usuários diferentes jogaram cada título.
# nunique() evita contar mais de uma vez o mesmo usuário para o mesmo jogo.
top_jogos_jogadores = (
    df_play.groupby('game_name')['user_id']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    x=top_jogos_jogadores.values,
    y=top_jogos_jogadores.index,
    hue=top_jogos_jogadores.index,
    palette="viridis",
    legend=False
)
plt.title('Top 10 Jogos por Quantidade de Jogadores Únicos', fontsize=18)
plt.xlabel('Número de Jogadores Únicos')
plt.ylabel('Jogo')
plt.tight_layout()
plt.show()

In [ ]:
# Soma as horas registradas para cada jogo e seleciona os dez maiores totais.
top_jogos_horas = (
    df_play.groupby('game_name')['hours']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 4))
sns.barplot(
    x=top_jogos_horas.values,
    y=top_jogos_horas.index,
    hue=top_jogos_horas.index,
    palette="rocket",
    legend=False
)
plt.ticklabel_format(style='plain', axis='x')
plt.title('Top 10 Jogos por Total de Horas Jogadas', fontsize=13)
plt.xlabel('Total de Horas Jogadas')
plt.ylabel('Jogo')
plt.tight_layout()
plt.show()

In [ ]:
# Para cada usuário, conta quantos jogos distintos aparecem em seus registros.
# nunique() conta títulos diferentes, mesmo que haja mais de um registro do mesmo jogo.
jogos_por_usuario = df_play.groupby('user_id')['game_name'].nunique()

# Agrupa os usuários pela quantidade de jogos distintos em seus registros.
distribuicao = jogos_por_usuario.value_counts().sort_index()

plt.figure(figsize=(10, 5))

# Representa a distribuição com uma área preenchida e uma linha.
plt.fill_between(distribuicao.index, distribuicao.values, alpha=0.4)
plt.plot(distribuicao.index, distribuicao.values, linewidth=2)

# Limita o eixo horizontal até 50 jogos para facilitar a visualização.
plt.xlim(1, 50)
plt.xticks([1, 10, 20, 30, 40, 50])

plt.title('Quantidade de Usuários por Número de Jogos Distintos', fontsize=14, pad=15)
plt.xlabel('Quantidade de Jogos Distintos')
plt.ylabel('Número de Usuários')

sns.despine()
plt.tight_layout()
plt.show()

# 5. Construção da Matriz Jogo-Usuário

In [ ]:
# Constrói uma matriz em que cada linha representa um jogo e cada coluna,
# um usuário. Os valores representam as horas transformadas (hours_weight).
# pivot_table utiliza a média como agregação padrão quando há mais de um
# registro para a mesma combinação de jogo e usuário.
# As combinações sem registro recebem zero.
matriz_jogos_usuarios = (
    df_play.pivot_table(
        index='game_name',
        columns='user_id',
        values='hours_weight',
        aggfunc='mean'
    )
    .fillna(0)
)

# A matriz pode ter muitos zeros, mas continua armazenada em um DataFrame denso
# do Pandas (não em uma estrutura de matriz esparsa do SciPy).
print("Formato da Matriz:", matriz_jogos_usuarios.shape)

# 6. Configuração e Treinamento do Modelo KNN (Scikit-Learn)

In [ ]:
# Configura o algoritmo de busca pelos jogos com vetores mais próximos.
# cosine: calcula a distância de cosseno entre os vetores de jogos.
# brute: compara o vetor consultado com os vetores disponíveis na matriz.
# n_neighbors=6 corresponde ao próprio jogo + cinco recomendações (padrão).
modelo_knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=6
)

# Ajusta o modelo aos vetores de todos os jogos da matriz.
modelo_knn.fit(matriz_jogos_usuarios)
print("Modelo treinado com sucesso!")

# 7. Função de Recomendação e Teste do Sistema

In [ ]:
# Solicita o nome do jogo que será pesquisado
nome_do_jogo = input("\nDigite o nome do jogo: ")

def recomendar_jogos(nome_do_jogo, modelo, matriz):

    # Verifica se o jogo existe na base de dados
    if nome_do_jogo not in matriz.index:
        return f"O jogo '{nome_do_jogo}' não foi encontrado na base de dados ou foi escrito de forma errada."

    # Localiza a posição do jogo na matriz
    indice_jogo = matriz.index.get_loc(nome_do_jogo)

    # Busca os jogos mais próximos utilizando o modelo KNN
    # reshape(1, -1) adapta os dados para o formato exigido pelo Scikit-Learn
    distancias, indices = modelo.kneighbors(matriz.iloc[indice_jogo, :].values.reshape(1, -1))

    print(f"Por você ter jogado ** {nome_do_jogo} **, recomendamos:\n")

    # Ignora o primeiro resultado, pois ele é o próprio jogo pesquisado
    for i in range(1, len(distancias.flatten())):
        jogo_recomendado = matriz.index[indices.flatten()[i]]
        distancia = distancias.flatten()[i]

        # Quanto menor a distância, maior a semelhança entre os jogos
        print(f"{i}. {jogo_recomendado} (Distância: {distancia:.3f})")


recomendar_jogos(nome_do_jogo, modelo_knn, matriz_jogos_usuarios)